# 1. Preparativos obligatorios

Nos hemos conectado a una GPU T4.

In [1]:
#Instalamos lubrerías necesarias
!pip install -q --upgrade datasets transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.0 MB/s eta 0:00:00


In [2]:
#Nos logueamos con nuestro token
from huggingface_hub import login
import os

token = os.getenv("HF_TOKEN")

login(token=token)

# 2. Misionees de Código

## Reto 1. El traductor universal

In [3]:
from transformers import AutoTokenizer

modelo_id = 'meta-llama/Meta-Llama-3.1-8B'

tokenizer = AutoTokenizer.from_pretrained(modelo_id, trust_remote_code=True)

frase = "Los LLMs no entienden palabras. Solo entienden matemáticas."

# Aplicamos el método encode()
tokens_ids = tokenizer.encode(frase)

print(tokens_ids)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[128000, 30696, 445, 11237, 82, 912, 1218, 72, 20468, 74304, 13, 36223, 1218, 72, 20468, 5634, 336, 92355, 13]


## Reto 2. Ingeniería Inversa

In [5]:
#Recuperamos la frase original
frase_orginal = tokenizer.decode(tokens_ids)

print(frase_orginal)

<|begin_of_text|>Los LLMs no entienden palabras. Solo entienden matemáticas.


In [9]:
# Para ver las piezas sueltas de forma individual
# Lo hago de esta forma porque el método .batch_decode(tokens) no devulve la salida que nosotros deseamos
piezas_reales = [tokenizer.decode([t]) for t in tokens_ids]
print(piezas_reales)

['<|begin_of_text|>', 'Los', ' L', 'LM', 's', ' no', ' ent', 'i', 'enden', ' palabras', '.', ' Solo', ' ent', 'i', 'enden', ' mat', 'em', 'áticas', '.']


1. ¿Qué token especial ha inyectado el modelo automáticamente al principio del texto?
- <|begin_of_text|>

2. Busca la palabra "matemáticas" en la lista devuelta. ¿En cuántos fragmentos se ha dividido
la palabra?
- La palabra se ha dividido en 3 tokens diferentes

## Reto 3: El diferctor y el Actor

In [11]:
#Cargamos el modelo que queremos utilizar
modelo_instruct_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer_instruct = AutoTokenizer.from_pretrained(modelo_instruct_id, trust_remote_code=True)

mensajes = [
    {"role": "system", "content": "Eres un profesor de Inteligencia Artificial muy sarcástico."},
    {"role": "user", "content": "¿Me explicas qué es exactamente un tensor?"}
]

# utilizamos .apply_chat_template
texto_final = tokenizer_instruct.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)

print(texto_final)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Eres un profesor de Inteligencia Artificial muy sarcástico.<|eot_id|><|start_header_id|>user<|end_header_id|>

¿Me explicas qué es exactamente un tensor?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




# 3. Choque de Titanes

In [17]:
# Cargamos los tokenizadores

modelos = {
    "Llama 3.1": "meta-llama/Meta-Llama-3.1-8B",
    "Qwen 2.5 Coder": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Phi-4": "microsoft/Phi-4-mini-instruct",
    "DeepSeek V3": "deepseek-ai/DeepSeek-V3"
}

tokenizadores = {}
for nombre, repo in modelos.items():
    tokenizadores[nombre] = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


In [13]:
# Bloque de código que vamos a analizar
codigo = "def hello_world(person):\n    print('Hello', person)"

# Comprobar la longitud de tokens
for nombre, tk in tokenizadores.items():
    longitud = len(tk.encode(codigo))
    print(f"Longitud en {nombre}: {longitud} tokens")

Longitud en Llama 3.1: 13 tokens
Longitud en Qwen 2.5 Coder: 12 tokens
Longitud en Phi-4: 12 tokens
Longitud en DeepSeek V3: 13 tokens


In [15]:
print("Autopsia de tokens (Qwen 2.5 Coder)")
qwen_tokenizer = tokenizadores["Qwen 2.5 Coder"]
tokens_qwen = qwen_tokenizer.encode(codigo)

for t in tokens_qwen:
    print(f"{t} = '{qwen_tokenizer.decode([t])}'")

Autopsia de tokens (Qwen 2.5 Coder)
750 = 'def'
23811 = ' hello'
31792 = '_world'
29766 = '(person'
982 = '):
'
262 = '   '
1173 = ' print'
492 = '(''
9707 = 'Hello'
516 = '','
1697 = ' person'
8 = ')'
